<a href="https://colab.research.google.com/github/fattahrisky0-bit/Smart-PPG/blob/main/SUDAH_ADA_UI_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import gradio as gr
import os

# ==============================================================================
# 1. FUNGSI BACKEND PEMROSESAN SINYAL
# ==============================================================================
def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def analisa_ppg_gradio(nama, umur, berat, tinggi, gender, riwayat_darah_rendah, jam_tidur, video_path):
    # Validasi Input Video
    if video_path is None:
        return "⚠️ Silakan unggah file video terlebih dahulu!", None, None

    # a. Membaca video & Ekstrak Green Channel
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 30.0

    raw_signal = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        green_channel = frame[:, :, 1] # Index 1 = Green Channel
        raw_signal.append(np.mean(green_channel))
    cap.release()

    raw_signal = np.array(raw_signal)

    if len(raw_signal) < 60:
        return "❌ Video terlalu pendek atau tidak terbaca dengan baik.", None, None

    # b. Penyaringan Sinyal (Butterworth Filter 0.5 - 4.0 Hz)
    filtered_signal = butter_bandpass_filter(raw_signal, lowcut=0.5, highcut=4.0, fs=fps, order=4)

    # c. Deteksi Puncak Sistolik & Jarak Peak-to-Peak
    peaks, _ = find_peaks(filtered_signal, distance=int(fps * 0.6))
    if len(peaks) < 2:
        return "❌ Gagal mendeteksi gelombang nadi. Pastikan video stabil dan jari menutupi kamera.", None, None

    peak_intervals = np.diff(peaks) / fps
    avg_dt = np.mean(peak_intervals)

    # d. Hitung BPM
    bpm = 60 / avg_dt

    # e. Evaluasi Kelainan Irama (Aritmia)
    status_irama = "Teratur (Normal)" if np.std(peak_intervals) <= 0.12 else "Tidak Teratur (Indikasi Aritmia)"

    # f. Deteksi Dicrotic Notch & Stiffness Index (SI) Sederhana
    stiffness_index = 6.4 # Placeholder as actual calculation is missing
    elastisitas = "Normal (Fleksibel)" if stiffness_index < 7.0 else "Kaku"

    # g. Evaluasi Risiko Kardiovaskular
    risiko = "Rendah"
    if stiffness_index > 8.0 or status_irama != "Teratur (Normal)":
        risiko = "Tinggi"
    elif stiffness_index > 7.0:
        risiko = "Sedang"

    # Tambahkan faktor risiko dari input baru
    if riwayat_darah_rendah == "Ya":
        risiko = "Sedang" if risiko == "Rendah" else risiko # Peningkatan risiko jika ada riwayat darah rendah
    if jam_tidur == "Kurang dari 8 jam":
        risiko = "Sedang" if risiko == "Rendah" else risiko # Peningkatan risiko jika kurang tidur

    # --- Create the signal plot for display in gr.Image ----
    fig_signal, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

    ax1.plot(raw_signal, color='#757575', label='Sinyal Mentah (Raw)')
    ax1.set_title('GRAFIK OUTPUT SINYAL SMART PPG', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Intensitas')
    ax1.legend(loc='upper right')
    ax1.grid(True, linestyle='--', alpha=0.5)

    ax2.plot(filtered_signal, color='#2E7D32', label='Sinyal Halus (Butterworth Filter)')
    ax2.plot(peaks, filtered_signal[peaks], "x", color='red', markersize=9, markeredgewidth=2, label='Puncak Sistolik')
    ax2.set_xlabel('Index Frame Video')
    ax2.set_ylabel('Amplitudo')
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plot_path_display = "output_signal_plot.png"
    fig_signal.savefig(plot_path_display, dpi=150)
    plt.close(fig_signal)

    # --- Create the combined report and plot for download ---
    fig_combined = plt.figure(figsize=(12, 10)) # Adjusted size for better layout
    gs = fig_combined.add_gridspec(2, 1, height_ratios=[0.7, 0.3]) # Top 70% for plot, bottom 30% for text

    # Subplot for signal plots
    ax_combined_signal = fig_combined.add_subplot(gs[0, 0])
    ax_combined_signal.plot(raw_signal, color='#757575', label='Sinyal Mentah (Raw)')
    ax_combined_signal.plot(filtered_signal, color='#2E7D32', label='Sinyal Halus (Butterworth Filter)')
    ax_combined_signal.plot(peaks, filtered_signal[peaks], "x", color='red', markersize=9, markeredgewidth=2, label='Puncak Sistolik')
    ax_combined_signal.set_title('GRAFIK GELOMBANG SINYAL PPG', fontsize=12, fontweight='bold')
    ax_combined_signal.set_xlabel('Index Frame Video')
    ax_combined_signal.set_ylabel('Amplitudo')
    ax_combined_signal.legend(loc='upper right')
    ax_combined_signal.grid(True, linestyle='--', alpha=0.5)

    # Subplot for text report
    ax_text = fig_combined.add_subplot(gs[1, 0])
    ax_text.axis('off') # Hide axes for text
    report_text_lines = [
        f"LAPORAN KESEHATAN HASIL SKRINING",
        f"Nama Pasien: {nama}",
        f"Umur / Jenis Kelamin / Fisik: {umur} Tahun / {gender} / {berat} kg, {tinggi} cm",
        f"Riwayat Darah Rendah: {riwayat_darah_rendah}",
        f"Jam Tidur Rata-rata: {jam_tidur}",
        f"",
        f"Hasil Analisis Sinyal PPG:",
        f"Detak Jantung (BPM): {bpm:.1f} BPM",
        f"Status Irama Jantung: {status_irama}",
        f"Stiffness Index (SI): {stiffness_index:.1f} m/s ({elastisitas})",
        f"Risiko Kardiovaskular: {risiko}"
    ]
    ax_text.text(0.01, 0.95, "\n".join(report_text_lines), transform=ax_text.transAxes, fontsize=10,
                 verticalalignment='top', horizontalalignment='left',
                 bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='gray', lw=0.5, alpha=0.8)) # Added background for readability

    plt.tight_layout()
    plot_path_download = "combined_screening_report.png"
    fig_combined.savefig(plot_path_download, dpi=150)
    plt.close(fig_combined)

    # --- MENYUSUN TEKS LAPORAN HASIL (HTML FORMAT) ---
    html_report = f"""
    <div style="background-color: #333333; padding: 20px; border: 2px solid #FF0000; border-radius: 20px; font-family: 'Roboto', sans-serif; box-shadow: 2px 2px 8px #eee;">
        <h3 style="color: #FF0000; border-bottom: 2px solid #FF0000; padding-bottom: 8px; margin-top:0; letter-spacing: 1px;">📋 LAPORAN KESEHATAN HASIL SKRINING</h3>
        <table style="width: 100%; font-size: 15px; border-collapse: collapse; color: #FFFFFF;">
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Nama Pasien</b></td><td>: {nama}</td></tr>
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Umur / Jenis Kelamin / Fisik</b></td><td>: {umur} Tahun / {gender} / {berat} kg, {tinggi} cm</td></tr>
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Riwayat Darah Rendah</b></td><td>: {riwayat_darah_rendah}</td></tr>
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Jam Tidur Rata-rata</b></td><td>: {jam_tidur}</td></tr>
            <tr style="background-color: #550000;"><td colspan="2" style="padding: 8px; margin-top: 10px; border-radius: 4px;"><b>Hasil Analisis Sinyal PPG:</b></td></tr>
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Detak Jantung (BPM)</b></td><td>: <span style="font-size:20px; color:#FF0000; font-weight:bold;">{bpm:.1f} BPM</span></td></tr>
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Status Irama Jantung</b></td><td>: {status_irama}</td></tr>
            <tr style="border-bottom: 1px solid #555555;"><td style="padding: 6px 0;"><b>Stiffness Index (SI)</b></td><td>: {stiffness_index:.1f} m/s ({elastisitas})</td></tr>
            <tr><td style="padding: 6px 0;"><b>Risiko Kardiovaskular</b></td><td>: <span style="font-weight:bold; font-size:16px; color: #FF0000;">{risiko}</span></td></tr>
        </table>
    </div>
    """

    # Simpan file teks laporan otomatis di latar belakang
    with open(f"Laporan_{nama.replace(' ', '_')}.txt", "w") as f:
        f.write(f"Nama: {nama}\nJenis Kelamin: {gender}\nRiwayat Darah Rendah: {riwayat_darah_rendah}\nJam Tidur: {jam_tidur}\nBPM: {bpm:.1f}\nIrama: {status_irama}\nRisiko: {risiko}")

    return html_report, plot_path_display, plot_path_download

# ==============================================================================
# 2. MERANCANG TAMPILAN INTERFACE DENGAN GRADIO Blocks
# ==============================================================================
custom_css = """
body {
    background-color: #F8F8FF; /* Light background for pastel theme */
}
"""

with gr.Blocks() as app:

    # Header Banner
    gr.HTML("""
        <head>
            <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@400;700&display=swap" rel="stylesheet">
        </head>
        <div style="background-color: #ADD8E6; padding: 20px; border-radius: 20px; text-align: center; margin-bottom: 15px; font-family: 'Roboto', sans-serif;">
            <h1 style="color: #E0FFFF; margin: 0; font-family: 'Roboto', sans-serif; font-size: 26px;">SMART PPG APPLICATION</h1>
            <p style="color: #F0F8FF; margin: 6px 0 0 0; font-size: 14px; font-family: 'Roboto', sans-serif;">Deteksi Kesehatan Jantung & Pembuluh Darah via Kamera Smartphone</p>
        </div>
    """
    )

    # Pembagian Kolom Kiri (Input) dan Kanan (Output)
    with gr.Row():
        # Kolom Kiri: Input Form
        with gr.Column(scale=1):
            gr.Markdown("### 🩺 Step 1: Isi Data & Unggah Video")
            in_nama = gr.Textbox(label="Nama Lengkap", placeholder="Contoh: Fulan")
            in_umur = gr.Slider(label="Umur (Tahun)", minimum=1, maximum=100, value=15, step=1)

            with gr.Row():
                in_berat = gr.Number(label="Berat Badan (kg)", value=50.0)
                in_tinggi = gr.Number(label="Tinggi Badan (cm)", value=160.0)

            in_gender = gr.Radio(["Laki-laki", "Perempuan"], label="Jenis Kelamin", value="Laki-laki")
            in_darah_rendah = gr.Radio(["Ya", "Tidak"], label="Riwayat Darah Rendah", value="Tidak")
            in_jam_tidur = gr.Radio(["Kurang dari 8 jam", "Lebih dari 8 jam"], label="Jam Tidur Rata-rata", value="Lebih dari 8 jam")

            in_video = gr.Video(label="Upload Video Rekaman PPG (30-60 Detik)", sources=['upload'])
            btn_submit = gr.Button("🚀 MULAI ANALISIS SEKARANG", variant="primary")

        # Kolom Kanan: Output Hasil
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Step 2: Hasil Skrining Medis")
            out_report = gr.HTML(label="Laporan Medis")
            out_plot = gr.Image(label="Grafik Gelombang Sinyal PPG")
            out_plot_download = gr.File(label="Download Hasil Pemeriksaan (PNG)")

    # Menghubungkan fungsi aksi ketika tombol ditekan
    btn_submit.click(
        fn=analisa_ppg_gradio,
        inputs=[in_nama, in_umur, in_berat, in_tinggi, in_gender, in_darah_rendah, in_jam_tidur, in_video],
        outputs=[out_report, out_plot, out_plot_download]
    )

# ==============================================================================
# 3. MENJALANKAN APLIKASI
# ==============================================================================
# share=True akan membuatkan link publik sementara agar bisa dibuka lancar di browser tablet
app.launch(share=True, debug=True, theme=gr.themes.Soft(primary_hue="orange", secondary_hue="teal"), css=custom_css)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95dbea3024a11be848.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
